In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import numba as na
import scipy.special as ss
import scipy.optimize as so
from helpers import compute_beta_norm
from functools import partial
# import mpi4py.futures
from multiprocessing import Pool


In [2]:
# (@) Configuration (target)
#
DEF_ASPACE_MIN = 0.0
DEF_ASPACE_MAX = 2.5
DEF_ASPACE_NUM = 101

DEF_GSPACE_MIN = 0.0
DEF_GSPACE_MAX = 3.0
DEF_GSPACE_NUM = 101

# (@) Configuration (Gaussian state search)
#
DEF_XSPACE_MIN = - 6.0
DEF_XSPACE_MAX = + 6.0
DEF_PSPACE_MIN = - 6.0
DEF_PSPACE_MAX = + 6.0
DEF_RSPACE_MIN = - 1.5
DEF_RSPACE_MAX = + 1.5

In [3]:
@na.njit(cache = True)
def catable_operator_gaussian_mean (parity, alpha, gamma, norm, beta, r,u,v,q):
    '''
    Computes the mean value of the operator (2) for arbitrary Gaussian state.

    Parameters (operator)
    ---------------------
    parity: np.integer (values -1, 1)
        Parity of the target coherent Schrodinger state.
    alpha: np.complex
        Amplitude of the target Schrodinger state.
    gamma: np.floating
        Sensitivity of the non-Gaussian fringe detection.

    Parameters (Gaussian state)
    ---------------------------
    r: np.floating
        Effective squeezing of the state.
    u: np.floating
        Mean quadrature (position) of the state.
    v: np.floating
        Mean quadrature (momentum) of the state.
    q: np.floating
        Rotation of the state.

    Returns
    -------
    np.floating
        A non-negative number representing the mean value.
    '''

    ep = np.exp(  4.0 * r)
    em = np.exp(- 4.0 * r)
    ct = np.cos(q)
    st = np.sin(q)

    A = (em * (ct ** 2) + ep * (st ** 2)) * 0.5
    B = (ep * (ct ** 2) + em * (st ** 2)) * 0.5
    C = (ep - em) * ct * st * 0.5

    uu = u ** 2
    vv = v ** 2
    uv = u * v

    x = np.real(beta)*np.sqrt(2)
    p = np.imag(beta)*np.sqrt(2)

    f1 = (uu + vv) ** 2
    f2 = vv * A + uu * B + 2 * (C ** 2) + 4 * (C * uv) + A * B
    f3 = vv * B + uu * A
    f4 = A ** 2 + B ** 2
    f5 = uu + vv + A + B + np.real(alpha ** 2) * (uu - vv - B + A) + 2 * np.imag(alpha ** 2) * (uv + C) - np.abs(alpha) ** 4
    
    f6 =   (x-u)**2*B + (p-v)**2*A - C*(p*x+x*p + 2*v*u-2*p*u-2*x*v)           #for kerr cats
    # f6 = 2 * C * uv - B * uu - A * vv                                        #for parity cats
    
    q1 = (f1 + 2 * f2 + 6 * f3 + 3 * f4) / 4 - f5 + 0.5
    
    q2 = gamma * (np.abs(norm) - parity * np.exp(-2* f6))                     #for kerr cats
    # q2 = gamma * (1 - parity * np.exp(2 * f6))                              #for parity cats
    
    return q1 + q2

In [12]:
@na.njit(cache = True) 
def infidel_gaussian (parity, alpha, r, u,v):
    '''
    Computes the infidelity between an axis-aligned coherent state and an
    arbitrary axis-aligned pure Gaussian state constructed by displacing a
    squeezed vacuum state.

    Parameters (target)
    -------------------
    parity: np.integer (values -1, 1)
        Parity of the target coherent Schrodinger state.
    alpha: np.complex
        Amplitude of the target coherent Schrodinger state.

    Parameters (Gaussian state)
    ---------------------------
    r: np.floating
        Effective squeezing of the state.
    u: np.floating
        Mean quadrature (position) of the state.
    v: np.floating
        Mean quadrature (momentum) of the state.

    Returns
    -------
    np.floating
        A non-negative number representing the mean value.
    '''

    # We want to avoid ZeroDivisionError raised by numba.
    #
    # Note: Setting the error_model parameter to 'numpy' did not seem to yield
    # the desired behavior.

    if not (np.abs(alpha) > 0.0):
        return np.nan

    X0 = np.real(alpha) * np.sqrt(2) 
    P0 = np.imag(alpha) * np.sqrt(2)

    VX = 0.5 * np.exp(- 2.0 * r)
    VP = 0.5 * np.exp(+ 2.0 * r)

    fid = _cat_gaussian_fidelity(parity, X0, P0, u, VX, v, VP)
    return 1 - np.real(fid)

# A fidelity of an arbitrary axis-aligned Gaussian state, characterized by its
# quadrature means and variances, with an axis aligned coherent cat state.

@na.njit(cache = True)
def _cat_gaussian_fidelity (Cs, X0, P0, GmX, GvX, GmP, GvP):
    '''
    Computes a fidelity between an axis-aligned coherent cat state
    and an arbitrary axis-aligned Gaussian state characterized by its
    quadrature means and variances. 

    (*) Please note that the variances of physical states must satisfy the
    uncertainty relations (Simon, 1994). 

    A pure axis-aligned Gaussian state can be understood as a displaced
    squeezed vacuum (Kral, 1990). Its quadrature variances satisfy 

                           var(X) * var(P) = (1 / 4). 

    It is consequently sufficient to specify only one of the variances.

    (*) Please note that the amplitude is defined as

                       alpha := (X0 + 1j * P0) / sqrt(2)

    in terms of the mean quadature values.

    Parameters (target)
    -------------------
    Cs : integer (values -1, 1)
        Parity of the CS state.
        Use +1 to indicate even parity.
        Use -1 to indicate odd parity.
    X0 : np.floating
        X0 component of the CS state.
    P0 : np.floating
        P0 component of the CS state.

    Parameters (Gaussian state)
    ---------------------------
    GmX : np.floating
        Mean of X of the Gaussian state.
    GvX : np.floating
        Variance of X of the Gaussian state.
    GmP : np.floating
        Mean of P of the Gaussian state.
    GvP : np.floating
        Variance of P of the Gaussian state.

    Returns
    -------
    np.floating
        Fidelity (overlap) of the CS state and the Gaussian state.
    '''

    VX = 1.0 + 2.0 * GvX
    VP = 1.0 + 2.0 * GvP
          
    eXP = np.exp(- X0 ** 2 - P0 ** 2)
    fSS = np.sqrt(VX) * np.sqrt(VP)
    f1E = 1
    # f1E = (1 + Cs * eXP)
    

    fPP = _fragment_same_sign(+1, X0, P0, GmX, VX, GmP, VP)
    fMM = _fragment_same_sign(-1, X0, P0, GmX, VX, GmP, VP)
    fPM = _fragment_diff_sign(+1, X0, P0, GmX, VX, GmP, VP)
    fMP = _fragment_diff_sign(-1, X0, P0, GmX, VX, GmP, VP)

    return (fPP + 1j*Cs * (fPM - fMP) * eXP + fMM) / fSS / f1E    #for kerr cats 
    # return (fPP + Cs * (fPM + fMP) * eXP + fMM) / fSS / f1E     #for parity cats 


@na.njit(cache = True)
def _fragment_same_sign (Fs, X0, P0, MX, VX, MP, VP):
    qX = (X0 - Fs * MX) ** 2
    qP = (P0 - Fs * MP) ** 2
    eX = np.exp(- qX / VX)
    eP = np.exp(- qP / VP)
    return eX * eP

@na.njit(cache = True)
def _fragment_diff_sign (Fs, X0, P0, MX, VX, MP, VP):
    qX = (MX - Fs * 1j * P0) ** 2
    qP = (MP + Fs * 1j * X0) ** 2
    eX = np.exp(- qX / VX)
    eP = np.exp(- qP / VP)
    return eX * eP

In [5]:
def minimize (objective, x0, bounds, samples):
    bounds = np.array(bounds)
    random = np.random.default_rng()
    points = random.uniform(bounds[:, 0], bounds[:, 1], size = (samples, np.size(x0)))

    result = _minimize_compare(np.inf, objective, x0, bounds)
    for y0 in points:
        result = _minimize_compare(result, objective, y0, bounds)
    return result

def _minimize_compare (current, * args):
    result = _minimize_wrapper(* args)
    if result.success:
        if result.fun < current:
            return result.fun
    return current

def _minimize_wrapper (objective, x0, bounds):
    return so.minimize(objective, x0, method = 'L-BFGS-B', bounds = bounds)


In [6]:
def objective_cat (alpha, norm, beta, gamma):
    def objective (x):
        return catable_operator_gaussian_mean(+1, alpha, gamma, norm, beta,
            x[0], 0, x[1] , 0)                     #be careful, when alpha real then catable_operator_gaussian_mean(+1, alpha, gamma, norm, beta,x[0], x[1], 0 , 0) 
    return objective

def solve_for_cat(pspace,gspace,index): 
    aspace,nspace,bspace = pspace
    aindex, gindex = index
    return index,minimize(
            objective_cat(aspace[aindex], nspace[aindex], bspace[aindex], gspace[gindex]),
            [ 0, np.sqrt(2) * aspace[aindex] ],
            [ (DEF_RSPACE_MIN, DEF_RSPACE_MAX), 
              (DEF_XSPACE_MIN, DEF_XSPACE_MAX)
            ], 1001) 


def objective_inf (alpha):
    def objective (x):
        return infidel_gaussian(+1, alpha,
            x[0], x[1], x[2])
    return objective

def solve_for_inf(aspace,index):
    return index,minimize(
            objective_inf(aspace[index]),
            [ 0, np.sqrt(2) * aspace[index], 0 ],
            [ (DEF_RSPACE_MIN, DEF_RSPACE_MAX), 
              (DEF_XSPACE_MIN, DEF_XSPACE_MAX), 
              (DEF_PSPACE_MIN, DEF_PSPACE_MAX) 
            ], 1001)
         
def dispatch_cat(pool, pspace, gspace, worker, path):
    aspace,_,_ = pspace
    result = np.zeros((aspace.size, gspace.size), dtype=np.float64)
    mapper = partial(worker, pspace, gspace)
    mapped = pool.map(mapper, np.ndindex(result.shape))

    for index, value in mapped:
        result[index] = value

    np.save(path, result)
    return result

def dispatch_inf(pool, aspace, worker, path):
    result = np.zeros(shape = aspace.size, dtype=np.float64)
    mapper = partial(worker, aspace)
    mapped = pool.map(mapper, np.arange(aspace.size))

    for index, value in mapped:
        result[index] = value

    np.save(path, result)
    return result

In [13]:
%%time
aspace = np.linspace(DEF_ASPACE_MIN, DEF_ASPACE_MAX, DEF_ASPACE_NUM)
aspace = aspace*1j     # when aspace real then changes required in objective_cat, if changed to aspace real then compute_beta_norm DOES NOT WORK

gspace = np.linspace(DEF_GSPACE_MIN, DEF_GSPACE_MAX, DEF_GSPACE_NUM)
np.save('results/comparison/aspace.npy', aspace)
np.save('results/comparison/gspace.npy', gspace)
theta = np.pi/2   # when theta 0 or pi then changes required in , _cat_gaussian_fidelity,catable_operator_gaussian_mean 
parity = 1

bspace = np.zeros_like(aspace)
nspace = np.zeros_like(aspace)

for i, alpha in enumerate(aspace):
    alpha_real = alpha.imag
    beta, norm = compute_beta_norm(alpha_real, parity, theta)     # compute_beta_norm works only for alpha imaginary but takes only real input
    bspace[i] = beta
    nspace[i] = norm
np.save('results/comparison/bspace.npy', bspace)
np.save('results/comparison/nspace.npy', nspace)

pspace = (aspace, nspace, bspace)

if __name__ == "__main__":
    with Pool(processes=24) as pool:
        inf_result = dispatch_inf(pool,aspace,solve_for_inf,"results/comparison/bestinf.npy")
        cat_result = dispatch_cat(pool,pspace,gspace,solve_for_cat,"results/comparison/bestcat.npy")

/cluster/modules/python-3.11-env.essentials/lib/python3.11/site-packages/scipy/_lib/array_api_compat/common/_aliases.py:239: ComplexWarning: Casting complex values to real discards the imaginary part
  return x.astype(dtype=dtype, copy=copy)
/cluster/modules/python-3.11-env.essentials/lib/python3.11/site-packages/scipy/_lib/array_api_compat/common/_aliases.py:239: ComplexWarning: Casting complex values to real discards the imaginary part
  return x.astype(dtype=dtype, copy=copy)
/cluster/modules/python-3.11-env.essentials/lib/python3.11/site-packages/scipy/_lib/array_api_compat/common/_aliases.py:239: ComplexWarning: Casting complex values to real discards the imaginary part
  return x.astype(dtype=dtype, copy=copy)
/cluster/modules/python-3.11-env.essentials/lib/python3.11/site-packages/scipy/_lib/array_api_compat/common/_aliases.py:239: ComplexWarning: Casting complex values to real discards the imaginary part
  return x.astype(dtype=dtype, copy=copy)
/cluster/modules/python-3.11-env

CPU times: user 160 ms, sys: 83.9 ms, total: 244 ms
Wall time: 13min 28s
